# Análise de notícias da PETR4 com Yahoo Finance

Este notebook converte o script original em uma execução **etapa por etapa**, para facilitar testes no Anaconda/Jupyter.

Fluxo do notebook:
1. Instalar e importar dependências
2. Definir configurações e dicionários
3. Criar utilitários de texto
4. Buscar notícias no Yahoo Finance
5. Calcular relevância e impacto
6. Rodar a análise final
7. Exportar resultados para CSV/JSON

**Observação:** a busca de notícias via `yfinance` pode ser intermitente. Se uma execução falhar, rode novamente a célula de busca.

## 1. Instalação das dependências

Se você ainda não instalou as bibliotecas no ambiente do Anaconda, rode a célula abaixo uma vez.
Se já estiver tudo instalado, pode pular esta etapa.

In [1]:
# Descomente a linha abaixo se precisar instalar as dependências no ambiente atual.
!pip install yfinance pandas

  Using cached yfinance-1.2.1-py2.py3-none-any.whl.metadata (6.1 kB)
  Using cached multitasking-0.0.12-py3-none-any.whl
  Using cached frozendict-2.4.7-py3-none-any.whl.metadata (23 kB)
  Using cached peewee-4.0.4-py3-none-any.whl.metadata (8.6 kB)
  Using cached curl_cffi-0.15.0-cp310-abi3-win_amd64.whl.metadata (18 kB)
  Using cached websockets-16.0-cp311-cp311-win_amd64.whl.metadata (7.0 kB)
  Using cached cffi-2.0.0-cp311-cp311-win_amd64.whl.metadata (2.6 kB)
Using cached yfinance-1.2.1-py2.py3-none-any.whl (130 kB)
Using cached curl_cffi-0.15.0-cp310-abi3-win_amd64.whl (1.7 MB)
Using cached frozendict-2.4.7-py3-none-any.whl (16 kB)
Using cached peewee-4.0.4-py3-none-any.whl (144 kB)
Using cached websockets-16.0-cp311-cp311-win_amd64.whl (178 kB)
Using cached cffi-2.0.0-cp311-cp311-win_amd64.whl (182 kB)
  Attempting uninstall: cffi
    Found existing installation: cffi 1.16.0
    Uninstalling cffi-1.16.0:
      Successfully uninstalled cffi-1.16.0


  You can safely remove it manually.


## 2. Imports

Aqui carregamos as bibliotecas usadas no protótipo.

In [2]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import yfinance as yf

## 3. Configuração do ativo e dicionários de apoio

Nesta etapa definimos:
- o perfil da ação acompanhada;
- os temas monitorados;
- os sinais heurísticos de impacto;
- uma lista simples de stopwords.

In [3]:
ACOES: Dict[str, Dict[str, Any]] = {
    "PETR4.SA": {
        "ticker": "PETR4.SA",
        "nome": "Petrobras",
        "empresa_aliases": [
            "petrobras", "petróleo brasileiro", "petroleo brasileiro", "pbr", "petr4"
        ],
        "temas": ["petroleo", "geopolitica", "governo", "cambio", "dividendos", "resultado"],
        "entidades": [
            "petrobras", "brent", "opep", "diesel", "gasolina", "pre-sal", "pré-sal",
            "refino", "combustível", "combustiveis", "produção", "producao"
        ],
        "descricao_semantica": (
            "Petrobras é uma petroleira brasileira sensível a petróleo Brent, OPEP, "
            "combustíveis, decisões do governo, dividendos, produção, refino, câmbio "
            "e geopolítica energética."
        ),
    }
}

TEMAS: Dict[str, List[str]] = {
    "petroleo": [
        "petróleo", "petroleo", "brent", "barril", "opep", "oferta", "produção", "producao",
        "refino", "combustível", "combustiveis", "diesel", "gasolina", "energia", "pré-sal", "pre-sal"
    ],
    "geopolitica": [
        "guerra", "conflito", "sanção", "sancao", "ataque", "tensão", "tensao",
        "bloqueio", "oriente médio", "oriente medio", "russia", "ucrânia", "ucrania",
        "israel", "irã", "ira", "estreito", "rotas marítimas", "rotas maritimas"
    ],
    "governo": [
        "governo", "presidente", "ministério", "ministerio", "regulação", "regulacao",
        "estatal", "política de preços", "politica de precos", "tributo", "imposto", "aneel", "anp"
    ],
    "cambio": ["dólar", "dolar", "câmbio", "cambio", "real", "moeda", "fx", "usd", "brl"],
    "dividendos": ["dividendo", "dividendos", "proventos", "jcp", "payout", "yield"],
    "resultado": [
        "balanço", "balanco", "resultado", "lucro", "ebitda", "receita", "guidance",
        "produção recorde", "producao recorde", "earnings"
    ],
}

SINAIS_IMPACTO = {
    "positivo": [
        "alta do petróleo", "alta do petroleo", "brent sobe", "corte de oferta", "dividendos",
        "lucro", "recorde de produção", "recorde de producao", "aumento de produção",
        "aumento de producao", "guidance positivo", "recompra", "descoberta"
    ],
    "negativo": [
        "queda do petróleo", "queda do petroleo", "brent cai", "intervenção do governo",
        "intervencao do governo", "controle de preços", "controle de precos", "acidente",
        "derramamento", "prejuízo", "prejuizo", "sanção", "sancao", "processo", "investigação",
        "investigacao", "redução de dividendos", "reducao de dividendos"
    ],
}

STOPWORDS = {
    "a", "o", "e", "de", "do", "da", "dos", "das", "em", "um", "uma", "para", "com",
    "por", "no", "na", "nos", "nas", "que", "os", "as", "ao", "à", "às", "ou", "se",
    "sobre", "mais", "menos", "entre", "após", "apos", "aponta", "diz", "segundo", "como",
    "yahoo", "finance", "reuters", "ap", "the", "and", "of", "in", "to", "for", "on", "at",
}

## 4. Classe de dados para cada notícia

Essa estrutura ajuda a deixar os itens de notícia mais organizados.

In [4]:
@dataclass
class NewsItem:
    title: str
    summary: str
    publisher: str
    link: str
    published_at: Optional[datetime]
    raw: Dict[str, Any]

## 5. Utilidades de texto

Essas funções:
- normalizam texto;
- quebram em tokens;
- geram uma lista simples de palavras-chave frequentes.

In [5]:
def normalize_text(text: str) -> str:
    text = (text or "").strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text

def tokenize(text: str) -> List[str]:
    text = normalize_text(text)
    tokens = re.findall(r"[\w\-áàâãéèêíïóôõöúçñ]+", text, flags=re.IGNORECASE)
    return [t for t in tokens if len(t) > 2 and t not in STOPWORDS]

def top_keywords_from_text(text: str, top_n: int = 12) -> List[str]:
    counts: Dict[str, int] = {}
    for token in tokenize(text):
        counts[token] = counts.get(token, 0) + 1
    ranked = sorted(counts.items(), key=lambda x: (-x[1], x[0]))
    return [w for w, _ in ranked[:top_n]]

## 6. Funções para buscar e interpretar notícias do Yahoo Finance

Aqui o notebook:
- busca notícias pelo ticker;
- tenta múltiplas rotas do `yfinance`;
- transforma o retorno bruto em objetos padronizados.

In [6]:
def _safe_ts_to_datetime(value: Any) -> Optional[datetime]:
    try:
        if value is None:
            return None
        return datetime.fromtimestamp(int(value), tz=timezone.utc)
    except Exception:
        return None

def _parse_news_item(item: Dict[str, Any]) -> Optional[NewsItem]:
    if not isinstance(item, dict):
        return None

    content = item.get("content") if isinstance(item.get("content"), dict) else item
    title = content.get("title") or item.get("title") or ""
    summary = content.get("summary") or item.get("summary") or ""
    publisher = content.get("provider") or content.get("publisher") or item.get("publisher") or ""

    link = ""
    canonical = content.get("canonicalUrl") if isinstance(content.get("canonicalUrl"), dict) else None
    if canonical:
        link = canonical.get("url") or ""
    if not link:
        clickthrough = content.get("clickThroughUrl") if isinstance(content.get("clickThroughUrl"), dict) else None
        if clickthrough:
            link = clickthrough.get("url") or ""
    if not link:
        link = content.get("link") or item.get("link") or ""

    published = content.get("pubDate") or item.get("providerPublishTime") or item.get("pubDate")
    published_at = _safe_ts_to_datetime(published)

    title = str(title).strip()
    summary = str(summary).strip()
    publisher = str(publisher).strip()
    link = str(link).strip()

    if not title:
        return None

    return NewsItem(
        title=title,
        summary=summary,
        publisher=publisher,
        link=link,
        published_at=published_at,
        raw=item,
    )

def fetch_yahoo_news(ticker: str, count: int = 20) -> List[NewsItem]:
    tk = yf.Ticker(ticker)
    raw_candidates: List[Dict[str, Any]] = []
    errors: List[str] = []

    for method_name in ("get_news", "news"):
        try:
            if method_name == "get_news":
                method = getattr(tk, "get_news", None)
                if callable(method):
                    result = method(count=count, tab="news")
                else:
                    continue
            else:
                result = getattr(tk, "news", None)

            if isinstance(result, list) and result:
                raw_candidates = result
                break
        except Exception as exc:
            errors.append(f"{method_name}: {exc}")

    parsed: List[NewsItem] = []
    seen: set[Tuple[str, str]] = set()

    for item in raw_candidates:
        parsed_item = _parse_news_item(item)
        if not parsed_item:
            continue
        dedupe_key = (parsed_item.title.lower(), parsed_item.link.lower())
        if dedupe_key in seen:
            continue
        seen.add(dedupe_key)
        parsed.append(parsed_item)

    if not parsed and errors:
        raise RuntimeError(
            "Falha ao obter notícias do Yahoo Finance via yfinance. "
            f"Detalhes: {' | '.join(errors)}"
        )

    return parsed[:count]

## 7. Teste rápido de coleta

Rode esta célula para confirmar se as notícias estão vindo do Yahoo Finance antes de seguir para a classificação.

In [7]:
ticker = "PETR4.SA"
count = 10

news_items = fetch_yahoo_news(ticker=ticker, count=count)
print(f"Total de notícias retornadas: {len(news_items)}")

for i, item in enumerate(news_items[:5], start=1):
    print(f"\n[{i}] {item.title}")
    print(f"Publisher: {item.publisher}")
    print(f"Data UTC: {item.published_at}")
    print(f"Link: {item.link}")
    print(f"Resumo: {item.summary[:250]}...")

Total de notícias retornadas: 10

[1] Subsea7 secures Sépia 2 field contract from Petrobras
Publisher: {'displayName': 'Offshore Technology', 'url': 'https://www.globaldata.com/', 'sourceId': 'offshore_technology_431'}
Data UTC: None
Link: https://www.offshore-technology.com/news/subsea7-sepia2-field-contract-petrobras/
Resumo: The Sépia 2 project is part of Brazil’s pre-salt expansion, a central focus for offshore investment....

[2] Sector Update: Energy Stocks Lean Lower Premarket Friday
Publisher: {'displayName': 'MT Newswires', 'url': 'https://www.mtnewswires.com/', 'sourceId': 'mt_newswires_premium_news_706'}
Data UTC: None
Link: https://finance.yahoo.com/sectors/energy/articles/sector-energy-stocks-lean-lower-132826838.html
Resumo: Energy stocks were leaning lower premarket Friday, with the State Street Energy Select Sector SPDR E...

[3] Can Petrobras' Pre-Salt Dominance Keep Driving Strong Growth? (Revised)
Publisher: {'displayName': 'Zacks', 'url': 'http://www.zacks.com/', 's

## 8. Funções de extração de temas, entidades, relevância e impacto

Agora entram as heurísticas da V1:
- temas detectados no texto;
- entidades ligadas ao ativo;
- score de relevância;
- rótulo de impacto provável.

In [8]:
def extract_themes(text: str) -> List[str]:
    text_n = normalize_text(text)
    found: List[str] = []
    for tema, palavras in TEMAS.items():
        if any(p in text_n for p in palavras):
            found.append(tema)
    return found

def extract_entities_simple(text: str, ticker: str) -> List[str]:
    text_n = normalize_text(text)
    entities = []
    perfil = ACOES[ticker]
    for ent in perfil["entidades"] + perfil["empresa_aliases"]:
        if ent in text_n:
            entities.append(ent)
    return sorted(set(entities))

def relevance_score(text: str, ticker: str) -> Tuple[int, List[str], List[str], List[str]]:
    text_n = normalize_text(text)
    perfil = ACOES[ticker]

    themes = extract_themes(text_n)
    entities = extract_entities_simple(text_n, ticker)
    top_keywords = top_keywords_from_text(text_n)

    score = 0

    for alias in perfil["empresa_aliases"]:
        if alias in text_n:
            score += 10
            break

    for ent in perfil["entidades"]:
        if ent in text_n:
            score += 4

    for theme in themes:
        if theme in perfil["temas"]:
            score += 3

    if "petroleo" in themes and "geopolitica" in themes:
        score += 4
    if "governo" in themes and any(a in text_n for a in perfil["empresa_aliases"]):
        score += 5
    if "dividendos" in themes and any(a in text_n for a in perfil["empresa_aliases"]):
        score += 5
    if "resultado" in themes and any(a in text_n for a in perfil["empresa_aliases"]):
        score += 5

    return score, themes, entities, top_keywords

def classify_relevance(score: int) -> str:
    if score >= 18:
        return "alta"
    if score >= 9:
        return "media"
    return "baixa"

def classify_impact(text: str, ticker: str, is_relevant: bool) -> Tuple[str, float, List[str], List[str]]:
    text_n = normalize_text(text)

    positive_hits = [term for term in SINAIS_IMPACTO["positivo"] if term in text_n]
    negative_hits = [term for term in SINAIS_IMPACTO["negativo"] if term in text_n]

    themes = extract_themes(text_n)
    if ticker == "PETR4.SA":
        if "petroleo" in themes and any(k in text_n for k in ["sobe", "alta", "dispara", "avança", "avanca"]):
            positive_hits.append("petróleo em alta")
        if "petroleo" in themes and any(k in text_n for k in ["cai", "queda", "recuo"]):
            negative_hits.append("petróleo em queda")
        if "governo" in themes and any(k in text_n for k in ["intervenção", "intervencao", "controle"]):
            negative_hits.append("pressão governamental")
        if "dividendos" in themes:
            positive_hits.append("tema dividendos")
        if "resultado" in themes and any(k in text_n for k in ["lucro", "recorde", "supera", "forte"]):
            positive_hits.append("resultado forte")
        if "resultado" in themes and any(k in text_n for k in ["prejuízo", "prejuizo", "fraco", "abaixo"]):
            negative_hits.append("resultado fraco")

    pos = len(set(positive_hits))
    neg = len(set(negative_hits))

    if not is_relevant:
        return "incerto", 0.15, sorted(set(positive_hits)), sorted(set(negative_hits))

    if pos == 0 and neg == 0:
        return "neutro", 0.35, [], []
    if pos > neg:
        conf = min(0.55 + (pos - neg) * 0.1, 0.9)
        return "positivo", conf, sorted(set(positive_hits)), sorted(set(negative_hits))
    if neg > pos:
        conf = min(0.55 + (neg - pos) * 0.1, 0.9)
        return "negativo", conf, sorted(set(positive_hits)), sorted(set(negative_hits))
    return "incerto", 0.4, sorted(set(positive_hits)), sorted(set(negative_hits))

## 9. Teste manual em uma notícia

Use esta etapa para inspecionar a heurística em um texto específico.

In [9]:
if news_items:
    exemplo = f"{news_items[0].title}. {news_items[0].summary}"
    score, themes, entities, keywords = relevance_score(exemplo, ticker)
    relevance = classify_relevance(score)
    impact, confidence, pos_hits, neg_hits = classify_impact(exemplo, ticker, is_relevant=(relevance != "baixa"))

    print("Título:", news_items[0].title)
    print("Score:", score)
    print("Relevância:", relevance)
    print("Temas:", themes)
    print("Entidades:", entities)
    print("Palavras-chave:", keywords)
    print("Impacto:", impact)
    print("Confiança:", confidence)
    print("Sinais positivos:", pos_hits)
    print("Sinais negativos:", neg_hits)
else:
    print("Nenhuma notícia carregada na etapa 7.")

Título: Subsea7 secures Sépia 2 field contract from Petrobras
Score: 21
Relevância: alta
Temas: ['petroleo']
Entidades: ['petrobras', 'pre-sal']
Palavras-chave: ['sépia', 'brazil', 'central', 'contract', 'expansion', 'field', 'focus', 'from', 'investment', 'offshore', 'part', 'petrobras']
Impacto: neutro
Confiança: 0.35
Sinais positivos: []
Sinais negativos: []


## 10. Pipeline principal de análise

Esta função junta todas as etapas e devolve um `DataFrame` final.

In [10]:
def analyze_news_for_ticker(ticker: str, count: int, query: Optional[str] = None) -> pd.DataFrame:
    if ticker not in ACOES:
        raise ValueError(f"Ticker '{ticker}' não está configurado no dicionário ACOES.")

    news_items = fetch_yahoo_news(ticker=ticker, count=count)
    rows: List[Dict[str, Any]] = []

    query_n = normalize_text(query) if query else ""

    for item in news_items:
        full_text = f"{item.title}. {item.summary}".strip()
        score, themes, entities, keywords = relevance_score(full_text, ticker)
        relevance = classify_relevance(score)

        if query_n and query_n not in normalize_text(full_text):
            if relevance == "baixa":
                continue

        impact, confidence, pos_hits, neg_hits = classify_impact(
            full_text,
            ticker=ticker,
            is_relevant=(relevance != "baixa"),
        )

        rows.append({
            "ticker": ticker,
            "empresa": ACOES[ticker]["nome"],
            "published_at_utc": item.published_at.isoformat() if item.published_at else None,
            "publisher": item.publisher,
            "title": item.title,
            "summary": item.summary,
            "link": item.link,
            "relevance_score": score,
            "relevance_label": relevance,
            "impact_label": impact,
            "impact_confidence": round(confidence, 3),
            "themes": ", ".join(themes),
            "entities": ", ".join(entities),
            "top_keywords": ", ".join(keywords),
            "positive_signals": ", ".join(pos_hits),
            "negative_signals": ", ".join(neg_hits),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df["published_dt"] = pd.to_datetime(df["published_at_utc"], utc=True, errors="coerce")
    df = df.sort_values(
        by=["relevance_score", "published_dt"],
        ascending=[False, False],
        kind="mergesort",
    ).drop(columns=["published_dt"])

    return df.reset_index(drop=True)

## 11. Rodar a análise completa

Aqui você executa a análise consolidada e já vê o resultado em tabela.

In [11]:
ticker = "PETR4.SA"
count = 20
query = None  # Exemplo: "Petrobras"

df_resultado = analyze_news_for_ticker(ticker=ticker, count=count, query=query)
print(f"Total de linhas analisadas: {len(df_resultado)}")
df_resultado.head(10)

Total de linhas analisadas: 20


,ticker,empresa,published_at_utc,publisher,title,summary,link,relevance_score,relevance_label,impact_label,impact_confidence,themes,entities,top_keywords,positive_signals,negative_signals
0,PETR4.SA,Petrobras,None,"{'displayName': 'Offshore Technology', 'url': ...",Subsea7 secures Sépia 2 field contract from Pe...,The Sépia 2 project is part of Brazil’s pre-sa...,https://www.offshore-technology.com/news/subse...,21,alta,neutro,0.35,petroleo,"petrobras, pre-sal","sépia, brazil, central, contract, expansion, f...",,
1,PETR4.SA,Petrobras,None,"{'displayName': 'Zacks', 'url': 'http://www.za...",Can Petrobras' Pre-Salt Dominance Keep Driving...,"PBR's pre-salt edge drives output growth, low ...",https://finance.yahoo.com/sectors/energy/artic...,21,alta,neutro,0.35,petroleo,"pbr, petrobras, pre-sal","growth, pre-salt, can, cash, costs, discoverie...",,
2,PETR4.SA,Petrobras,None,"{'displayName': 'Zacks', 'url': 'http://www.za...",Can Petrobras' Pre-Salt Dominance Keep Driving...,"PBR's pre-salt edge drives output growth, low ...",https://finance.yahoo.com/sectors/energy/artic...,21,alta,neutro,0.35,petroleo,"pbr, petrobras, pre-sal","growth, pre-salt, can, cash, costs, discoverie...",,
3,PETR4.SA,Petrobras,None,"{'displayName': 'Insider Monkey', 'url': 'http...",Is Petroleo Brasileiro (PBR) One of the Must-B...,Petroleo Brasileiro (NYSE:PBR) is among the mu...,https://finance.yahoo.com/markets/stocks/artic...,21,alta,positivo,0.65,"petroleo, dividendos","pbr, petroleo brasileiro","brasileiro, pbr, petroleo, invest, must-buy, n...",tema dividendos,
4,PETR4.SA,Petrobras,None,"{'displayName': 'Insider Monkey', 'url': 'http...",Jim Cramer on Petrobras: “I Wish I Could Have ...,Petróleo Brasileiro S.A. – Petrobras (NYSE:PBR...,https://finance.yahoo.com/markets/stocks/artic...,17,media,neutro,0.35,petroleo,"pbr, petrobras, petróleo brasileiro","cramer, stock, could, earlier, have, jim, petr...",,
5,PETR4.SA,Petrobras,None,"{'displayName': 'Oilprice.com', 'url': 'http:/...",Petrobras Moves to Reclaim Full Control of Key...,Petrobras has signed a $450 million agreement ...,https://finance.yahoo.com/sectors/energy/artic...,14,media,neutro,0.35,,petrobras,"full, petrobras, 450, agreement, assets, basin...",,
6,PETR4.SA,Petrobras,None,"{'displayName': 'Insider Monkey', 'url': 'http...",What the Petrobras Contract Means for Valaris ...,Valaris Limited (NYSE:VAL) is among the most p...,https://finance.yahoo.com/markets/stocks/artic...,14,media,neutro,0.35,,petrobras,"valaris, limited, val, contract, nyse, petrobr...",,
7,PETR4.SA,Petrobras,None,"{'displayName': 'Zacks', 'url': 'http://www.za...",Petrobras (PBR) Stock Falls Amid Market Uptick...,Petrobras (PBR) closed the most recent trading...,https://finance.yahoo.com/markets/stocks/artic...,14,media,neutro,0.35,,"pbr, petrobras","pbr, petrobras, trading, amid, closed, day, fa...",,
8,PETR4.SA,Petrobras,None,"{'displayName': 'Zacks', 'url': 'http://www.za...",Petrobras Extends Offshore Drilling Deals With...,PBR extends major offshore drilling contracts ...,https://finance.yahoo.com/sectors/energy/artic...,14,media,neutro,0.35,,"pbr, petrobras","drilling, extends, offshore, seadrill, valaris...",,
9,PETR4.SA,Petrobras,None,"{'displayName': 'Oilprice.com', 'url': 'http:/...",Petrobras Shakes Up Leadership Amid Governance...,Petrobras has implemented a series of leadersh...,https://finance.yahoo.com/sectors/energy/artic...,14,media,neutro,0.35,,petrobras,"governance, its, leadership, petrobras, transi...",,


## 12. Filtrar notícias mais relevantes

Essa etapa ajuda a inspecionar só o que ficou com relevância alta ou média.

In [12]:
if not df_resultado.empty:
    df_filtrado = df_resultado[df_resultado["relevance_label"].isin(["alta", "media"])].copy()
    df_filtrado[[
        "published_at_utc", "publisher", "title",
        "relevance_score", "relevance_label",
        "impact_label", "impact_confidence", "themes"
    ]].head(20)
else:
    print("Nenhum resultado para filtrar.")

## 13. Exportar os resultados

Salva os dados em CSV e, se quiser, também em JSON.

In [13]:
csv_path = "petr4_news_analysis.csv"
json_path = "petr4_news_analysis.json"

if not df_resultado.empty:
    df_resultado.to_csv(csv_path, index=False, encoding="utf-8-sig")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(df_resultado.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

    print("CSV salvo em:", csv_path)
    print("JSON salvo em:", json_path)
else:
    print("Nada para exportar.")

CSV salvo em: petr4_news_analysis.csv
JSON salvo em: petr4_news_analysis.json


## 14. Próximos passos sugeridos

Depois de validar este notebook, as evoluções mais naturais são:
- incluir mais ações além de PETR4;
- mover os perfis de ações para um arquivo de configuração;
- adicionar embeddings semânticos;
- trocar parte das heurísticas por um classificador treinado;
- criar uma interface com Streamlit.